In [19]:
import pandas as pd
from pathlib import Path
import json
import warnings
import os
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.signal import find_peaks
from scipy.ndimage import label as nd_label
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings('ignore')

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

CURRENT_CWD = Path.cwd()

if CURRENT_CWD.name == 'notebooks':
    PROJECT_ROOT = CURRENT_CWD.parent
    NOTEBOOK_OUTPUT_DIR = CURRENT_CWD
else:
    PROJECT_ROOT = CURRENT_CWD
    NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'notebooks'

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
SERIES_DIR = DATA_DIR / 'series'
ANNOTATIONS_FILE = DATA_DIR / 'data.json'

PLOTS_DIR = NOTEBOOK_OUTPUT_DIR / 'plots'
RESULTS_DIR = NOTEBOOK_OUTPUT_DIR / 'results'

for path in [PLOTS_DIR / 'base', PLOTS_DIR / 'baseline', RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

In [9]:
with open(DATA_DIR / 'data.json', 'r') as f:
    annotations_data = json.load(f)

print(f"Records loaded: {len(annotations_data)}")

def load_series(filename):
    """Loads time series data from txt/csv."""
    filepath = SERIES_DIR / f"{filename.split('-')[1].split('.')[0]}.txt"
    try:
        df = pd.read_csv(filepath, sep=';', parse_dates=['Time'])
        df['Speed_2'] = pd.to_numeric(df['Speed_2'], errors='coerce')
        return df
    except FileNotFoundError:
        print(f"Warning: File {filepath} not found.")
        return pd.DataFrame({'Speed_2': []})
    
def extract_annotations(item):
    """Parses Label Studio JSON export."""
    results = []
    if 'annotations' in item and len(item['annotations']) > 0:
        ann = item['annotations'][0]
        if 'result' in ann:
            for r in ann['result']:
                if r['type'] == 'timeserieslabels':
                    results.append({
                        'start': r['value']['start'],
                        'end': r['value']['end'],
                        'label': r['value']['timeserieslabels'][0]
                    })
    return results

all_segments = []

for item in annotations_data:
    file_id = item['file_upload'].split('-')[1].split('.')[0]
    df = load_series(item['file_upload'])
    if df.empty: continue
        
    annotations = extract_annotations(item)
    
    for ann in annotations:
        segment_data = df.iloc[ann['start']:ann['end']+1].copy()
        all_segments.append({
            'file_id': file_id,
            'label': ann['label'],
            'start': ann['start'],
            'end': ann['end'],
            'duration': ann['end'] - ann['start'] + 1,
            'data': segment_data
        })

print(f"Total segments extracted: {len(all_segments)}")

Records loaded: 16
Total segments extracted: 556


In [10]:
label_counts = pd.Series([s['label'] for s in all_segments]).value_counts()
print("\nClass Distribution:")
print(label_counts)

fig, axes = plt.subplots(len(annotations_data), 1, figsize=(20, 4 * len(annotations_data)))
if len(annotations_data) == 1:
    axes = [axes]

for idx, item in enumerate(annotations_data):
    file_id = item['file_upload'].split('-')[1].split('.')[0]
    df = load_series(item['file_upload'])
    if df.empty: continue
        
    annotations = extract_annotations(item)
    
    ax = axes[idx]
    ax.plot(df.index, df['Speed_2'], linewidth=0.8, color='gray', alpha=0.7, label='Speed')
    
    colors = {
        '1_Successful_Packing': '#28a745', 
        '2_Struggling_Process': '#ffc107', 
        '3_Fake_Launch': '#dc3545'         
    }
    
    for ann in annotations:
        color = colors.get(ann['label'], 'blue')
        ax.axvspan(ann['start'], ann['end'], alpha=0.3, color=color)
        
        # Add label text
        mid = (ann['start'] + ann['end']) / 2
        ax.text(mid, df['Speed_2'].max() * 0.9, ann['label'].split('_')[0], 
                ha='center', fontsize=8, weight='bold', color='black')
    
    ax.set_title(f"File: {file_id}", fontsize=12, weight='bold')
    ax.set_ylabel('Speed')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'base' / 'ground_truth_visualization.png', dpi=150, bbox_inches='tight')
plt.close()

print("Saved ground truth visualizations.")


Class Distribution:
1_Successful_Packing    358
3_Fake_Launch           198
Name: count, dtype: int64
Saved ground truth visualizations.


In [11]:
stats_by_label = {}

for label in label_counts.index:
    label_segments = [s for s in all_segments if s['label'] == label]
    
    durations = [s['duration'] for s in label_segments]
    max_speeds = [s['data']['Speed_2'].max() for s in label_segments]
    mean_speeds = [s['data']['Speed_2'].mean() for s in label_segments]
    zero_counts = [np.sum(s['data']['Speed_2'] == 0) for s in label_segments]
    zero_ratios = [zc / dur for zc, dur in zip(zero_counts, durations)]
    
    stats_by_label[label] = {
        'count': len(label_segments),
        'duration_mean': np.mean(durations),
        'duration_std': np.std(durations),
        'duration_min': np.min(durations),
        'duration_max': np.max(durations),
        'max_speed_mean': np.mean(max_speeds),
        'mean_speed_mean': np.mean(mean_speeds),
        'zero_ratio_mean': np.mean(zero_ratios),
        'zero_ratio_std': np.std(zero_ratios),
    }

stats_df = pd.DataFrame(stats_by_label).T
print("\nClass Statistics:")
print(stats_df.round(2))

stats_df.to_csv(RESULTS_DIR / 'class_statistics.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for label in label_counts.index:
    data = [s['duration'] for s in all_segments if s['label'] == label]
    axes[0].hist(data, alpha=0.6, label=label, bins=15, density=True)
axes[0].set_title('Duration Density')
axes[0].legend()

for label in label_counts.index:
    data = [s['data']['Speed_2'].max() for s in all_segments if s['label'] == label]
    axes[1].hist(data, alpha=0.6, label=label, bins=15, density=True)
axes[1].set_title('Max Speed Density')

for label in label_counts.index:
    data = [np.sum(s['data']['Speed_2'] == 0) / s['duration'] for s in all_segments if s['label'] == label]
    axes[2].hist(data, alpha=0.6, label=label, bins=15, density=True)
axes[2].set_title('Zero Ratio Density')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'base' / 'distributions.png', dpi=150)
plt.close()


Class Statistics:
                      count  duration_mean  duration_std  duration_min  \
1_Successful_Packing  358.0        1935.92        552.85         450.0   
3_Fake_Launch         198.0         378.98        278.73          59.0   

                      duration_max  max_speed_mean  mean_speed_mean  \
1_Successful_Packing        4263.0          221.54            64.20   
3_Fake_Launch               1617.0          160.63            33.74   

                      zero_ratio_mean  zero_ratio_std  
1_Successful_Packing             0.17            0.09  
3_Fake_Launch                    0.48            0.17  


In [12]:
def create_dataset(file_items, window_sizes=[10, 30, 100, 300, 500, 1000]):
    """
    Generates rolling window features for Time Series.
    Returns X (features), y (labels), and raw speeds for post-processing.
    """
    X_list = []
    y_list = []
    raw_speeds = []
    
    print("Generating advanced features...")
    
    for item in file_items:
        df = load_series(item['file_upload'])
        if df.empty: continue
            
        speed = df['Speed_2'].values
        
        # 0: Noise, 1: Successful Packing, 2: Fake/Issue
        y = np.zeros(len(speed), dtype=int)
        annotations = extract_annotations(item)
        
        for ann in annotations:
            label_code = 0
            if '1_Successful' in ann['label']: 
                label_code = 1
            elif '3_Fake' in ann['label'] or '2_Struggling' in ann['label']: 
                
                label_code = 2
            
            if label_code > 0:
                y[ann['start']:ann['end']+1] = label_code
        
        features = pd.DataFrame({'speed': speed})
        
        features['speed_lag_50'] = features['speed'].shift(50).fillna(0)
        
        for w in window_sizes:
            roll = features['speed'].rolling(w, center=True, min_periods=1)
            
            features[f'roll_mean_{w}'] = roll.mean()
            features[f'roll_max_{w}'] = roll.max()
            features[f'roll_std_{w}'] = roll.std()
            features[f'zero_ratio_{w}'] = (features['speed'] == 0).rolling(w, center=True, min_periods=1).mean()

        features = features.fillna(0)
        
        X_list.append(features)
        y_list.append(y)
        raw_speeds.append(speed)

    if not X_list:
        raise ValueError("No data loaded.")

    X_full = pd.concat(X_list, ignore_index=True)
    y_full = np.concatenate(y_list)
    
    return X_full, y_full, X_list, y_list, raw_speeds

train_files, test_files = train_test_split(annotations_data, test_size=0.25, random_state=42)

print(f"Train files: {len(train_files)}")
print(f"Test files: {len(test_files)}")

X_train, y_train, _, _, _ = create_dataset(train_files)
_, _, X_test_list, y_test_list, raw_speeds_test = create_dataset(test_files)

print(f"Train dataset shape: {X_train.shape}")

Train files: 12
Test files: 4
Generating advanced features...
Generating advanced features...
Train dataset shape: (908825, 26)


In [13]:
def calculate_hard_constraints(file_list):
    """Calculates physical limits from training data."""
    durations = []
    for item in file_list:
        anns = extract_annotations(item)
        for ann in anns:
            if '1_Successful' in ann['label']:
                durations.append(ann['end'] - ann['start'])
    
    if not durations:
        return 100 
        
    return np.array(durations)

train_durations = calculate_hard_constraints(train_files)

MIN_PACKING_LIMIT = int(np.percentile(train_durations, 1))

print(f"\nDuration Statistics (Train):")
print(f"  Min: {np.min(train_durations)}")
print(f"  1st Percentile: {MIN_PACKING_LIMIT}")
print(f"  Mean: {np.mean(train_durations):.0f}")
print(f"  Max: {np.max(train_durations)}")
print(f"Constraint set: Events shorter than {MIN_PACKING_LIMIT} samples will be ignored.")


Duration Statistics (Train):
  Min: 460
  1st Percentile: 580
  Mean: 1969
  Max: 4262
Constraint set: Events shorter than 580 samples will be ignored.


In [14]:
def smart_event_counter_v3(probs_pack, raw_speed, threshold=0.5, min_duration=100, gap_tolerance=10):
    mask = (probs_pack >= threshold).astype(int)
    
    # 2. Hard Cut based on physical machine stop
    # If speed <= 2 for more than gap_tolerance, it is a separator
    is_stopped = (raw_speed <= 2).astype(int)
    stopped_labels, num_stops = nd_label(is_stopped)
    
    refined_mask = mask.copy()
    
    for i in range(1, num_stops + 1):
        # Clear mask where machine is stopped
        if np.sum(stopped_labels == i) >= gap_tolerance:
            refined_mask[stopped_labels == i] = 0
            
    # 3. Filter short noise
    labeled_array, num_features = nd_label(refined_mask)
    valid_segments = []
    
    for i in range(1, num_features + 1):
        where = np.where(labeled_array == i)[0]
        length = len(where)
        
        if length >= min_duration:
            valid_segments.append((where[0], where[-1]))
            
    return len(valid_segments), valid_segments

In [15]:
print("Starting Hyperparameter Tuning (Grid Search)...")

param_grid = [
    {'n_estimators': [100], 'max_depth': [15, 25], 'min_samples_leaf': [20, 50]}
]

best_mae = float('inf')
best_model = None
best_params = {}

import itertools
keys = param_grid[0].keys()
values = param_grid[0].values()
combinations = list(itertools.product(*values))

# GAP_TOLERANCE for validation (keep it strict)
VAL_GAP_TOLERANCE = 5 

for i, comb in enumerate(combinations):
    params = dict(zip(keys, comb))
    print(f"Testing params: {params} ...", end=" ")
    
    model = RandomForestClassifier(
        **params, 
        class_weight='balanced', 
        n_jobs=-1, 
        random_state=42
    )
    model.fit(X_train, y_train)
    
    current_total_mae = 0
    
    for idx, df_test in enumerate(X_test_list):
        y_true = y_test_list[idx]
        raw_spd = raw_speeds_test[idx]
        
        probs = model.predict_proba(df_test)[:, 1]
        
        n_pred, _ = smart_event_counter_v3(probs, raw_spd, threshold=0.5, 
                                          min_duration=MIN_PACKING_LIMIT, 
                                          gap_tolerance=VAL_GAP_TOLERANCE)
        
        true_mask = (y_true == 1).astype(int)
        n_true, _ = smart_event_counter_v3(true_mask, raw_spd, threshold=0.5, 
                                          min_duration=MIN_PACKING_LIMIT, 
                                          gap_tolerance=VAL_GAP_TOLERANCE)
        
        current_total_mae += abs(n_pred - n_true)
        
    print(f"MAE: {current_total_mae}")
    
    if current_total_mae < best_mae:
        best_mae = current_total_mae
        best_model = model
        best_params = params

print(f"\nBest Parameters: {best_params}")
print(f"Best MAE: {best_mae}")

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=feature_importance.head(20))
plt.title('Top 20 Feature Importance')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'baseline' / 'feature_importance.png')
plt.close()
print("Feature importance saved.")

Starting Hyperparameter Tuning (Grid Search)...
Testing params: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 20} ... 

MAE: 0
Testing params: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 50} ... MAE: 0
Testing params: {'n_estimators': 100, 'max_depth': 25, 'min_samples_leaf': 20} ... MAE: 0
Testing params: {'n_estimators': 100, 'max_depth': 25, 'min_samples_leaf': 50} ... MAE: 0

Best Parameters: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 20}
Best MAE: 0
Feature importance saved.


In [16]:
print("Optimizing Probability Threshold...")

thresholds = np.arange(0.35, 0.70, 0.05)
best_thresh_mae = float('inf')
best_thresh = 0.5

FINAL_GAP_TOLERANCE = 5 

for th in thresholds:
    total_mae = 0
    for idx, df_test in enumerate(X_test_list):
        y_true = y_test_list[idx]
        raw_spd = raw_speeds_test[idx]
        probs = best_model.predict_proba(df_test)[:, 1]
        
        n_pred, _ = smart_event_counter_v3(probs, raw_spd, threshold=th, 
                                          min_duration=MIN_PACKING_LIMIT, 
                                          gap_tolerance=FINAL_GAP_TOLERANCE)
        
        true_mask = (y_true == 1).astype(int)
        n_true, _ = smart_event_counter_v3(true_mask, raw_spd, threshold=0.5, 
                                          min_duration=MIN_PACKING_LIMIT, 
                                          gap_tolerance=FINAL_GAP_TOLERANCE)
        
        total_mae += abs(n_pred - n_true)
        
    if total_mae < best_thresh_mae:
        best_thresh_mae = total_mae
        best_thresh = th

print(f"Best Threshold: {best_thresh:.2f} (MAE: {best_thresh_mae})")

Optimizing Probability Threshold...
Best Threshold: 0.35 (MAE: 0)


In [17]:
def calculate_event_metrics(pred_segs, true_segs):
    true_positives = 0
    
    for p_s, p_e in pred_segs:
        matched = False
        for t_s, t_e in true_segs:
    
            inter_start = max(p_s, t_s)
            inter_end = min(p_e, t_e)
            if inter_end > inter_start:
    
                overlap_len = inter_end - inter_start
                if overlap_len / (p_e - p_s) > 0.2 or overlap_len / (t_e - t_s) > 0.2:
                    matched = True
                    break
        if matched:
            true_positives += 1
            
    precision = true_positives / len(pred_segs) if len(pred_segs) > 0 else 0
    
    found_trues = 0
    for t_s, t_e in true_segs:
        matched = False
        for p_s, p_e in pred_segs:
            inter_start = max(p_s, t_s)
            inter_end = min(p_e, t_e)
            if inter_end > inter_start:
                 overlap_len = inter_end - inter_start
                 if overlap_len / (t_e - t_s) > 0.2:
                     matched = True
                     break
        if matched:
            found_trues += 1
            
    recall = found_trues / len(true_segs) if len(true_segs) > 0 else 0
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

In [18]:
def plot_professional_result(file_id, raw_speed, true_segs, pred_segs, ignored_segs, fake_segs, metrics):
    fig, (ax_speed, ax_timeline) = plt.subplots(2, 1, figsize=(20, 8), 
                                                sharex=True, 
                                                gridspec_kw={'height_ratios': [2.5, 1], 'hspace': 0.05})
    
    ax_speed.plot(raw_speed, color='#2c3e50', linewidth=1, alpha=0.8, label='Speed Signal')
    ax_speed.fill_between(range(len(raw_speed)), raw_speed, color='#2c3e50', alpha=0.05)
    
    
    for s, e in pred_segs:
        ax_speed.axvline(s, color='#007bff', alpha=0.1, linestyle='--')
        ax_speed.axvline(e, color='#007bff', alpha=0.1, linestyle='--')

    ax_speed.set_ylabel('Speed', fontsize=10, weight='bold')
    title = (f"File: {file_id} | True: {metrics['n_true']} | Pred: {metrics['n_pred']} | "
             f"Diff: {metrics['diff']} | F1: {metrics['f1']:.2f}")
    ax_speed.set_title(title, fontsize=14, weight='bold', pad=15)
    ax_speed.grid(True, axis='y', alpha=0.3, linestyle=':')
    
    y_true = 3
    y_pred = 2
    y_fake = 1
    y_noise = 0
    bar_height = 0.6
    
    for s, e in true_segs:
        ax_timeline.broken_barh([(s, e-s)], (y_true - bar_height/2, bar_height), 
                                facecolors='#28a745', edgecolors='#1e7e34')
        
    for s, e in pred_segs:
        ax_timeline.broken_barh([(s, e-s)], (y_pred - bar_height/2, bar_height), 
                                facecolors='#007bff', edgecolors='#0056b3')
        
    for s, e in fake_segs:
        ax_timeline.broken_barh([(s, e-s)], (y_fake - bar_height/2, bar_height), 
                                facecolors='#dc3545', edgecolors='#a71d2a')

    for s, e in ignored_segs:
        ax_timeline.broken_barh([(s, e-s)], (y_noise - bar_height/2, bar_height), 
                                facecolors='#ecf0f1', edgecolors='#95a5a6', hatch='///')

    ax_timeline.set_yticks([y_noise, y_fake, y_pred, y_true])
    ax_timeline.set_yticklabels(['Ignored', 'Issues/Fake', 'Prediction', 'Ground Truth'], 
                                fontsize=9, weight='bold')
    ax_timeline.set_ylim(-0.5, 3.5)
    ax_timeline.set_xlabel('Time (Samples)', fontsize=10)
    ax_timeline.grid(True, axis='x', alpha=0.5, linestyle='--')
    
    for ax in [ax_speed, ax_timeline]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
    plt.savefig(PLOTS_DIR / 'baseline' / f'final_viz_{file_id}.png', dpi=150, bbox_inches='tight')
    plt.close()


def get_raw_segments(mask):
    """Просто извлекает сегменты из бинарной маски без обработки скоростью."""
    labeled, num = nd_label(mask)
    segments = []
    for i in range(1, num + 1):
        where = np.where(labeled == i)[0]
        segments.append((where[0], where[-1]))
    return len(segments), segments


print("\n" + "="*110)
print(f"{'File':<10} | {'True':<5} | {'Pred':<5} | {'Diff':<5} | {'Prec':<6} | {'Rec':<6} | {'F1':<6} | {'Status'}")
print("="*110)

total_mae = 0
metrics_summary = []

for idx, df_test in enumerate(X_test_list):
    file_id = test_files[idx]['file_upload'].split('-')[1].split('.')[0]
    y_true = y_test_list[idx]
    raw_spd = raw_speeds_test[idx]
    
    probas = best_model.predict_proba(df_test)
    probs_pack = probas[:, 1]
    
    if probas.shape[1] > 2:
        probs_fake = probas[:, 2]
    else:
        probs_fake = np.zeros(len(probs_pack))
    
    n_pred, segs_pred = smart_event_counter_v3(probs_pack, raw_spd, 
                                               threshold=best_thresh, 
                                               min_duration=MIN_PACKING_LIMIT, 
                                               gap_tolerance=FINAL_GAP_TOLERANCE)
    
    true_mask = (y_true == 1).astype(int)
    n_true, segs_true = get_raw_segments(true_mask) 
    
    _, raw_segs = smart_event_counter_v3(probs_pack, raw_spd, threshold=best_thresh, 
                                         min_duration=0, gap_tolerance=FINAL_GAP_TOLERANCE)
    ignored_segs = [s for s in raw_segs if (s[1]-s[0]) < MIN_PACKING_LIMIT]
    
    n_fake, segs_fake = smart_event_counter_v3(probs_fake, raw_spd, threshold=0.4, 
                                               min_duration=50, gap_tolerance=10)
    
    diff = n_pred - n_true
    total_mae += abs(diff)
    status = "OK" if diff == 0 else ("OVER" if diff > 0 else "UNDER")
    
    prec, rec, f1 = calculate_event_metrics(segs_pred, segs_true)
    
    print(f"{file_id:<10} | {n_true:<5} | {n_pred:<5} | {diff:<5} | {prec:.2f}   | {rec:.2f}   | {f1:.2f}   | {status}")
    
    metrics = {'n_true': n_true, 'n_pred': n_pred, 'diff': diff, 'status': status, 'f1': f1}
    metrics_summary.append(metrics)
    
    plot_professional_result(file_id, raw_spd, segs_true, segs_pred, ignored_segs, segs_fake, metrics)

pd.DataFrame(metrics_summary).to_csv(RESULTS_DIR / 'final_metrics_summary.csv', index=False)

print("="*110)
print(f"TOTAL MAE: {total_mae}")


File       | True  | Pred  | Diff  | Prec   | Rec    | F1     | Status
1025       | 17    | 17    | 0     | 1.00   | 1.00   | 1.00   | OK
1029       | 31    | 29    | -2    | 1.00   | 0.94   | 0.97   | UNDER
1102       | 9     | 9     | 0     | 1.00   | 1.00   | 1.00   | OK
1111       | 12    | 10    | -2    | 1.00   | 0.83   | 0.91   | UNDER
TOTAL MAE: 4
